# Wake Word Quickstart
> **Zero to ONNX in one notebook.**  
> Synthesise a dataset with TTS, train a compact wake-word model, export to ONNX, and verify inference — all from a single wake-word string.

---

## What this notebook does

| Step | Cell | What happens |
|------|------|--------------|
| 1. Config | 2 | Set wake word, tier, epochs, MLflow URI |
| 2. Install | 3 | Install Python dependencies; auto-detect platform |
| 3. MLflow | 4 | Inject credentials from Kaggle Secrets (or skip) |
| 4. Dataset | 5 | Download / synthesise a labelled audio dataset |
| 5. Train | 6 | Train with `train_from_wakeword()` |
| 6. ONNX check | 7 | Verify both ONNX files exist; print sizes |
| 7. Inference | 8 | Quick sanity-check with `OnnxWakeWordInferencer` |
| 8. Summary | 9 | Print final metrics |

---

## Quick start

**Minimum viable run:** click **Run All** — outputs land in `./ww_output/`.

**Change the wake word:** set the `WAKE_WORD` env var (Kaggle Secrets, shell export, or edit Cell 2 directly).

---

## Configuration reference

| Variable | Default | Purpose |
|----------|---------|---------|
| `WAKE_WORD` | `hey jarvis` | Target phrase — any language |
| `OUTPUT_DIR` | `./ww_output` | All outputs land here |
| `TIER` | `small` | Architecture tier — see tier table |
| `EPOCHS` | `50` | Training epochs |
| `BATCH_SIZE` | `32` | Training batch size (reduce to 16 on CPU) |
| `N_POSITIVE` | `500` | TTS samples to synthesise |
| `LANG` | `en` | BCP-47 language for TTS |
| `ADVERSARIAL` | `true` | Add phonetically-similar hard negatives |
| `DOWNLOAD_AUGMENT` | `true` | Download bg-noise / music / RIR from HF |
| `DEVICE` | `auto` | `auto`, `cpu`, or `cuda` |
| `SEED` | `42` | Random seed |
| `MLFLOW_URI` | *(empty)* | MLflow tracking URI (optional) |
| `MLFLOW_SECRET` | `MLFLOW_TOKEN` | Kaggle Secret name for the MLflow token |
| `CUSTOM_TRAIN_CSV` | *(empty)* | BYO dataset — skips TTS synthesis |
| `CUSTOM_TEST_CSV` | *(empty)* | BYO test split (auto 80/20 if absent) |
| `REUSE_DATASET` | `true` | Skip datagen if dataset already exists |
| `VC_REFS_DIR` | *(empty)* | Voice-clone donor WAV directory |

---

## Tier reference

| Tier | Extractor | Params | Target hardware |
|------|-----------|--------|-----------------|
| `micro` | MFCC-20 | ~50 K | MCU / RPi Zero |
| `small` | MFCC-40 | ~200 K | RPi 3/4 |
| `filterbank_small` | FilterBank | ~250 K | Embedded SBC |
| `gammatone_small` | Gammatone | ~250 K | Embedded SBC |
| `sincnet_small` | SincNet | ~350 K | Low-power CPU |
| `delta_micro` | MFCC+Δ+ΔΔ | ~75 K | MCU with more flash |

---

## Outputs

```
ww_output/
├── dataset/
│   ├── train/metadata.csv
│   └── test/metadata.csv
└── model/
    ├── best_f1.pt
    ├── best_f1.onnx             # classifier head
    └── best_f1_featurizer.onnx  # feature extractor (required for inference)
```

Both ONNX files are required for inference:
```python
from ww_trainer.inference import OnnxWakeWordInferencer
model = OnnxWakeWordInferencer("best_f1_featurizer.onnx", "best_f1.onnx")
score = model.infer(wav_array)  # float in [0, 1]
```

---

## Platform notes

| Platform | Recommended settings |
|----------|---------------------|
| **Kaggle GPU T4** | `DEVICE=cuda`, `N_POSITIVE=500`, `DOWNLOAD_AUGMENT=true`; store MLflow token in Secrets |
| **Google Colab** | `DEVICE=cuda`; set `OUTPUT_DIR=/content/drive/MyDrive/ww_output` for persistence |
| **Paperspace** | `DEVICE=cuda`, `OUTPUT_DIR=/notebooks/ww_output` |
| **Local CPU** | reduce `N_POSITIVE=100`, `BATCH_SIZE=16`, `DOWNLOAD_AUGMENT=false` for speed |

## Cell 2 — Configuration
This is the **only cell you need to edit**. Every value can also be set as an environment variable or Kaggle Secret.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD        = os.environ.get("WAKE_WORD",        "hey jarvis")
OUTPUT_DIR       = os.environ.get("OUTPUT_DIR",       "./ww_output")
DEVICE           = os.environ.get("DEVICE",           "auto")
SEED             = int(os.environ.get("SEED",         "42"))

# ── Model ─────────────────────────────────────────────────────────────────────
TIER             = os.environ.get("TIER",             "small")
EPOCHS           = int(os.environ.get("EPOCHS",       "50"))
BATCH_SIZE       = int(os.environ.get("BATCH_SIZE",   "32"))

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",   "500"))
LANG             = os.environ.get("LANG",             "en")
ADVERSARIAL      = os.environ.get("ADVERSARIAL",      "true").lower() == "true"
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT", "true").lower() == "true"
REUSE_DATASET    = os.environ.get("REUSE_DATASET",    "true").lower() == "true"
CUSTOM_TRAIN_CSV = os.environ.get("CUSTOM_TRAIN_CSV", "")
CUSTOM_TEST_CSV  = os.environ.get("CUSTOM_TEST_CSV",  "")
VC_REFS_DIR      = os.environ.get("VC_REFS_DIR",      "") or None

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI       = os.environ.get("MLFLOW_URI",       "")
MLFLOW_SECRET    = os.environ.get("MLFLOW_SECRET",    "MLFLOW_TOKEN")

## Cell 3 — Install & platform detection
Installs all required packages. **Safe to skip** if `ww_trainer` is already installed.

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r} | Tier: {TIER}")

## Cell 4 — MLflow setup
Injects credentials from Kaggle Secrets. Safe to skip if MLflow is not needed — training proceeds without it.

In [ ]:
import os

# Inject MLflow token from Kaggle Secrets (no-op on other platforms)
if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"No Kaggle secret '{MLFLOW_SECRET}' found — MLflow will run unauthenticated or skip. ({e})")

if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    print(f"MLflow URI: {MLFLOW_URI}")
else:
    print("MLFLOW_URI not set — experiment tracking disabled")

## Cell 5 — Dataset

Three modes, tried in order:

1. **BYO CSV** (`CUSTOM_TRAIN_CSV` set) — use your existing `path,label` CSV directly.
2. **Auto** (default) — for known wake words, positives are downloaded from HuggingFace; for unknown ones they are synthesised via TTS (edge-tts).

`REUSE_DATASET=true` (default) means re-running this cell never re-synthesises audio that already exists.

In [ ]:
import shutil
from pathlib import Path

# Disk space guard
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, f"Only {free_gb:.1f} GB free — refusing to start. Free up space first."
print(f"Disk free: {free_gb:.1f} GB")

if CUSTOM_TRAIN_CSV:
    # ── BYO mode ──────────────────────────────────────────────────────────────
    from ww_trainer.utils import read_dataset_csv
    import random
    train_csv = Path(CUSTOM_TRAIN_CSV)
    if CUSTOM_TEST_CSV:
        test_csv = Path(CUSTOM_TEST_CSV)
    else:
        # Auto 80/20 split — written once, reused on subsequent runs
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED)
            random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            import csv
            for path, rows_slice in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(path, "w", newline="") as f:
                    csv.writer(f).writerows(rows_slice)
            print(f"Split written: {len(rows[:cut])} train / {len(rows[cut:])} test")
        train_csv, test_csv = split_train, split_test
    print(f"BYO mode: train={train_csv}, test={test_csv}")
    _byo_mode = True
else:
    # ── Auto / TTS mode ───────────────────────────────────────────────────────
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline

    datagen_cfg = DatagenConfig(
        wake_word=WAKE_WORD,
        output_dir=Path(OUTPUT_DIR) / "dataset",
        n_positive=N_POSITIVE,
        lang=LANG,
        adversarial=ADVERSARIAL,
        vad_trim=True,
        vc_refs_dir=VC_REFS_DIR,
        download_augmentation=DOWNLOAD_AUGMENT,
        seed=SEED,
    )

    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        print(f"Reusing existing dataset at {dataset_dir}")
        from ww_trainer.datagen import DatagenResult, normalize_wake_word
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
    else:
        print(f"Running datagen pipeline for '{WAKE_WORD}'...")
        _dr = run_datagen_pipeline(datagen_cfg)

    train_csv = _dr.train_csv
    test_csv  = _dr.test_csv
    _byo_mode = False
    print(f"Dataset ready: train={train_csv}, test={test_csv}")

## Cell 6 — Train

Calls `train_from_wakeword()` using the dataset prepared above. The best checkpoint and both ONNX files are saved to `OUTPUT_DIR/model/`.

In [ ]:
from ww_trainer.quickstart import train_from_wakeword, QuickstartConfig
from pathlib import Path

# Build augmentation kwargs from datagen result (if auto mode)
_aug_kwargs = {}
if not _byo_mode and hasattr(_dr, "bg_noise_dir") and _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
    _aug_kwargs["bg_noise_folder"] = str(_dr.bg_noise_dir)
if not _byo_mode and hasattr(_dr, "music_dir") and _dr.music_dir and Path(_dr.music_dir).exists():
    _aug_kwargs["music_folder"] = str(_dr.music_dir)
if not _byo_mode and hasattr(_dr, "rir_dir") and _dr.rir_dir and Path(_dr.rir_dir).exists():
    _aug_kwargs["rir_folder"] = str(_dr.rir_dir)

print(f"Training: tier={TIER!r}, epochs={EPOCHS}, batch_size={BATCH_SIZE}, device={DEVICE!r}")
print(f"Augmentation: {list(_aug_kwargs.keys()) or 'none'}")

result = train_from_wakeword(
    WAKE_WORD,
    OUTPUT_DIR,
    tier=TIER,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    reuse_dataset=True,   # dataset already prepared above
    **_aug_kwargs,
)

print("\nTraining complete.")
print(f"  best_onnx_path : {result.best_onnx_path}")
print(f"  best_model_path: {result.best_model_path}")
print(f"  metrics        : {result.metrics}")

## Cell 7 — ONNX export check

Verifies that both required ONNX files exist and are non-empty. Both are required for inference.

In [ ]:
from pathlib import Path

model_dir = Path(OUTPUT_DIR) / "model"
_head_onnx  = model_dir / "best_f1.onnx"
_feat_onnx  = model_dir / "best_f1_featurizer.onnx"

def _check(path, label):
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  OK  {label}: {path.name} ({size_kb:.0f} KB)")
    else:
        print(f"  MISSING  {label}: {path}")

print("ONNX files:")
_check(_feat_onnx, "featurizer")
_check(_head_onnx, "classifier  ")

assert _head_onnx.exists() and _feat_onnx.exists(), \
    "One or both ONNX files missing — check training logs above."

## Cell 8 — Inference sanity check

Loads both ONNX files and runs inference on a real positive sample from the test set to confirm the model predicts above 0.5.

In [ ]:
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

inferencer = OnnxWakeWordInferencer(str(_feat_onnx), str(_head_onnx))

# Find a positive sample from the test set
_pos_path = None
with open(test_csv) as f:
    for row in csv.reader(f):
        if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
            _pos_path = row[0]
            break

if _pos_path:
    wav, sr = torchaudio.load(_pos_path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav_np = wav.mean(0).numpy().astype(np.float32)
    score = inferencer.infer(wav_np)
    print(f"Positive sample: {Path(_pos_path).name}")
    print(f"  Confidence score: {score:.4f}  ({'PASS' if score > 0.5 else 'LOW — model may need more training'})")
else:
    print("No positive sample found in test CSV — skipping inference check.")

## Cell 9 — Summary

In [ ]:
from pathlib import Path

print("=" * 60)
print(f"Wake word  : {WAKE_WORD!r}")
print(f"Tier       : {TIER}")
print(f"Epochs     : {EPOCHS}")
print(f"Metrics    : {result.metrics}")
print()
print("Output files:")
for f in sorted(Path(OUTPUT_DIR).rglob("*.onnx")):
    print(f"  {f.relative_to(OUTPUT_DIR)}  ({f.stat().st_size/1024:.0f} KB)")
print("=" * 60)
print()
print("To test on a WAV file locally:")
print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
print(f"      --featurizer {_feat_onnx} \\")
print(f"      --model      {_head_onnx} \\")
print(f"      --audio      sample.wav")